# WebSocket streaming

For low-latency workflows, subscribe to Pyth's WebSocket stream at `/v1/stream`.
Each subscription returns a continuous sequence of `streamUpdated` frames containing
`parsed.priceFeeds` plus any requested on-chain payloads.

Pyth Pro requires you to maintain subscriptions to **at least two endpoints** for
reliability — this notebook shows a single connection for clarity, but production code
should round-robin across `pyth-lazer-0/1/2.dourolabs.app`.


In [ ]:
!pip install -q 'pyth-pandas[ws]' python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from pyth_pandas.ws import PythWebSocket

## Subscribe and drain a few frames

Open a session, subscribe with an arbitrary client-chosen `subscription_id`, then pull
messages off the queue until you've seen a few updates.

In [ ]:
ws = PythWebSocket()
frames = []

with ws.open() as session:
    session.subscribe(
        subscription_id=1,
        symbols=["Crypto.BTC/USD", "Crypto.ETH/USD"],
        properties=["price", "confidence", "exponent", "publisherCount"],
        formats=[],
        channel="real_time",
    )

    while len(frames) < 5:
        msg = session.recv(timeout=10)
        frames.append(msg)
        print(msg.get("type"), "—", len(msg.get("parsed", {}).get("priceFeeds", [])), "feeds")

    session.unsubscribe(subscription_id=1)

## Turn frames into a DataFrame

Each `streamUpdated` frame is identical in shape to a `/latest_price` response, so you
can reuse the same mantissa-to-decimal conversion.

In [ ]:
import pandas as pd

rows = []
for frame in frames:
    if frame.get("type") != "streamUpdated":
        continue
    parsed = frame.get("parsed") or {}
    ts = int(parsed.get("timestampUs", 0))
    for feed in parsed.get("priceFeeds", []):
        row = dict(feed)
        row["timestampUs"] = ts
        rows.append(row)

df = pd.DataFrame(rows)
if not df.empty:
    for col in ("price", "confidence", "exponent"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["exponent"] = df["exponent"].astype("Float64")
    df["humanPrice"] = df["price"] * 10 ** df["exponent"]
    df["ts"] = pd.to_datetime(df["timestampUs"], unit="us", utc=True)
df

## Notes

- The sync client exposes `PythWebSocket.open().subscribe(...) / recv() / unsubscribe(...)`.
- An async variant (`AsyncPythWebSocket`) is available for asyncio apps — subscribe once
  and iterate the session with `async for frame in session:`.
- Subscriptions can be scoped to `real_time` or fixed-rate channels
  (`fixed_rate@50ms / @200ms / @1000ms`).